# Retail Intelligence Platform - SQL Seller Analysis

This notebook builds the **seller analytics layer** for the Retail Intelligence Platform using PostgreSQL.

## Objectives
- calculate seller-level revenue and order contribution
- identify top sellers by revenue and volume
- analyze seller distribution by state and city
- evaluate seller review performance
- analyze seller delivery performance and late delivery exposure
- export clean seller outputs to `sql/outputs/`

## PostgreSQL tables used
- `olist_order_items`
- `olist_orders`
- `olist_sellers`
- `olist_order_reviews`

## Reusable view used
- `vw_sales_fact`

In [31]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from pathlib import Path

# --------------------------------------------------
# PostgreSQL connection
# --------------------------------------------------
DB_USER = "postgres"
DB_PASSWORD = quote_plus("1234")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "retail_intelligence"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("PostgreSQL engine created successfully.")

PostgreSQL engine created successfully.


## 1. Validate base seller table
We first confirm that seller records exist and the database connection is working correctly.

In [32]:
seller_test_query = """
SELECT COUNT(*) AS total_sellers
FROM olist_sellers;
"""

df_seller_test = pd.read_sql(seller_test_query, engine)
df_seller_test

,total_sellers
0,3095


## 2. Create seller summary view

This section creates a reusable **seller summary view** containing:
- seller revenue
- total orders served
- total items sold
- average item price
- seller geography

In [33]:
create_seller_view_sql = """
DROP VIEW IF EXISTS vw_seller_performance_summary;

CREATE VIEW vw_seller_performance_summary AS
WITH delivered_orders AS (
    SELECT
        o.order_id,
        o.order_purchase_timestamp::date AS order_purchase_date,
        o.order_delivered_customer_date::date AS delivered_date,
        o.order_estimated_delivery_date::date AS estimated_date
    FROM olist_orders o
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp IS NOT NULL
      AND o.order_delivered_customer_date IS NOT NULL
      AND o.order_estimated_delivery_date IS NOT NULL
),

order_reviews_agg AS (
    SELECT
        order_id,
        AVG(review_score) AS review_score
    FROM olist_order_reviews
    GROUP BY order_id
),

seller_order_base AS (
    SELECT
        oi.seller_id,
        oi.order_id,
        oi.order_item_id,
        oi.price,
        oi.freight_value,
        d.order_purchase_date,
        d.delivered_date,
        d.estimated_date,
        r.review_score,
        (d.delivered_date - d.order_purchase_date) AS delivery_days,
        CASE
            WHEN d.delivered_date > d.estimated_date THEN (d.delivered_date - d.estimated_date)
            ELSE 0
        END AS delay_days,
        CASE
            WHEN d.delivered_date > d.estimated_date THEN 1
            ELSE 0
        END AS is_late
    FROM olist_order_items oi
    JOIN delivered_orders d
        ON oi.order_id = d.order_id
    LEFT JOIN order_reviews_agg r
        ON oi.order_id = r.order_id
)

SELECT
    seller_id,
    COUNT(DISTINCT order_id) AS orders,
    COUNT(order_item_id) AS items,
    ROUND(SUM(COALESCE(price, 0))::numeric, 2) AS revenue,
    ROUND(SUM(COALESCE(freight_value, 0))::numeric, 2) AS freight,
    ROUND(AVG(review_score)::numeric, 4) AS avg_review_score,
    ROUND(AVG(delivery_days)::numeric, 4) AS avg_delivery_days,
    ROUND(AVG(delay_days)::numeric, 4) AS avg_delay_days,
    ROUND(100.0 * AVG(is_late)::numeric, 4) AS late_delivery_rate
FROM seller_order_base
GROUP BY seller_id;
"""

with engine.begin() as conn:
    conn.exec_driver_sql(create_seller_view_sql)

print("vw_seller_performance_summary created successfully.")

vw_seller_performance_summary created successfully.


## 3. Preview seller summary view

In [34]:
seller_summary_preview_query = """
SELECT *
FROM vw_seller_summary
LIMIT 10;
"""

df_seller_summary_preview = pd.read_sql(seller_summary_preview_query, engine)
df_seller_summary_preview

,seller_id,seller_city,seller_state,total_orders,total_items_sold,seller_revenue,seller_freight,avg_item_price
0,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP,3,3,2685.00,63.06,895.000000
1,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES,200,239,25080.03,8854.14,104.937364
2,001e6ad469a905060d959994f1b41e4f,sao goncalo,RJ,1,1,250.00,17.94,250.000000
3,002100f778ceb8431b7a1020ff7ab48f,franca,SP,51,55,1234.50,793.66,22.445455
4,003554e2dce176b5555353e4f3555ac8,goiania,GO,1,1,120.00,19.38,120.000000
5,004c9cd9d87a3c30c522c48c4fc07416,ibitinga,SP,158,170,19712.71,3551.23,115.957118
6,00720abe85ba0859807595bbf045a33b,guarulhos,SP,13,26,1007.50,315.98,38.750000
7,00ab3eff1b5192e5f1a63bcecfee11c8,sao paulo,SP,1,1,98.00,12.08,98.000000
8,00d8b143d12632bad99c0ad66ad52825,belo horizonte,MG,1,1,86.00,51.10,86.000000
9,00ee68308b45bc5e2660cd833c3f81cc,sao paulo,SP,135,172,20260.00,3180.66,117.790698


## 4. Seller KPI Summary

This section calculates the main seller KPIs:
- total sellers
- total seller revenue
- average revenue per seller
- average orders per seller
- average items sold per seller
- average freight per seller

In [35]:
seller_kpi_query = """
SELECT
    COUNT(*) AS total_sellers,
    ROUND(SUM(revenue)::numeric, 2) AS seller_revenue,
    ROUND(AVG(revenue)::numeric, 4) AS avg_revenue_per_seller,
    ROUND(AVG(orders)::numeric, 4) AS avg_orders_per_seller,
    ROUND(AVG(avg_review_score)::numeric, 4) AS avg_seller_review_score,
    ROUND(MAX(revenue)::numeric, 2) AS top_seller_revenue
FROM vw_seller_performance_summary;
"""

df_seller_kpis = pd.read_sql(seller_kpi_query, engine)
df_seller_kpis

,total_sellers,seller_revenue,avg_revenue_per_seller,avg_orders_per_seller,avg_seller_review_score,top_seller_revenue
0,2970,13220248.93,4451.2623,32.933,4.1505,226987.93


## 5. Top sellers by revenue
This section identifies the highest revenue-generating sellers.

In [36]:
top_sellers_revenue_query = """
SELECT
    seller_id,
    seller_city,
    seller_state,
    total_orders,
    total_items_sold,
    ROUND(seller_revenue::numeric, 2) AS seller_revenue,
    ROUND(avg_item_price::numeric, 2) AS avg_item_price
FROM vw_seller_summary
ORDER BY seller_revenue DESC
LIMIT 20;
"""

df_top_sellers_revenue = pd.read_sql(top_sellers_revenue_query, engine)
df_top_sellers_revenue

,seller_id,seller_city,seller_state,total_orders,total_items_sold,seller_revenue,avg_item_price
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1132,1156,229472.63,198.51
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,358,410,222776.05,543.36
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1806,1987,200472.92,100.89
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,585,586,194042.03,331.13
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,982,1364,187923.89,137.77
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,336,340,176431.87,518.92
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1314,1551,160236.57,103.31
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1160,1171,141745.53,121.05
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,915,1428,138968.55,97.32
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1287,1499,135171.70,90.17


## 6. Top sellers by order volume
This section identifies sellers serving the highest number of distinct orders.

In [37]:
top_sellers_orders_query = """
SELECT
    seller_id,
    seller_city,
    seller_state,
    total_orders,
    total_items_sold,
    ROUND(seller_revenue::numeric, 2) AS seller_revenue
FROM vw_seller_summary
ORDER BY total_orders DESC, seller_revenue DESC
LIMIT 20;
"""

df_top_sellers_orders = pd.read_sql(top_sellers_orders_query, engine)
df_top_sellers_orders

,seller_id,seller_city,seller_state,total_orders,total_items_sold,seller_revenue
0,6560211a19b47992c3666cc44a7e94c0,sao paulo,SP,1854,2033,123304.83
1,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1806,1987,200472.92
2,cc419e0650a3c5ba77189a1882b7556a,santo andre,SP,1706,1775,104288.42
3,1f50f920176fa81dab994f9023523100,sao jose do rio preto,SP,1404,1931,106939.21
4,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1314,1551,160236.57
5,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1287,1499,135171.70
6,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1160,1171,141745.53
7,ea8482cd71df3c1969d7b9473ff13abc,sao paulo,SP,1146,1203,37177.52
8,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1132,1156,229472.63
9,3d871de0142ce09b7081e2b9d1733cb1,campo limpo paulista,SP,1080,1147,94914.20


In [38]:
seller_performance_query = """
SELECT
    seller_id,
    orders,
    items,
    ROUND(revenue::numeric, 2) AS revenue,
    ROUND(freight::numeric, 2) AS freight,
    ROUND(avg_review_score::numeric, 4) AS avg_review_score,
    ROUND(avg_delivery_days::numeric, 4) AS avg_delivery_days,
    ROUND(avg_delay_days::numeric, 4) AS avg_delay_days,
    ROUND(late_delivery_rate::numeric, 4) AS late_delivery_rate
FROM vw_seller_performance_summary
ORDER BY revenue DESC;
"""

df_seller_performance = pd.read_sql(seller_performance_query, engine)
df_seller_performance

,seller_id,orders,items,revenue,freight,avg_review_score,avg_delivery_days,avg_delay_days,late_delivery_rate
0,4869f7a5dfa277a7dca6462dcf3b52b2,1124,1148,226987.93,20019.13,4.1395,14.9364,0.9739,10.5401
1,53243585a1d6dc2643021fd1853d8905,348,400,217940.44,12856.58,4.1281,13.2900,0.2975,3.0000
2,4a3ca9315b744ce9f8e9374361493884,1772,1949,196882.12,34338.31,3.8282,14.3699,1.0713,9.6973
3,fa1c13f2614d7b5c4749cbc52fecda94,578,579,190917.14,9916.36,4.3739,13.2953,0.9965,9.1537
4,7c67e1448b00f6e969d365cea6b010ab,973,1355,186570.05,51236.64,3.3474,22.3336,0.9653,8.8561
...,...,...,...,...,...,...,...,...,...
2965,4965a7002cca77301c82d3f91b82e1a9,1,1,8.49,7.87,5.0000,6.0000,0.0000,0.0000
2966,ad14615bdd492b01b0d97922e87cb87f,1,1,8.25,10.96,5.0000,14.0000,0.0000,0.0000
2967,702835e4b785b67a084280efca355756,1,1,7.60,10.96,5.0000,2.0000,0.0000,0.0000
2968,1fa2d3def6adfa70e58c276bb64fe5bb,1,1,6.90,9.00,1.0000,6.0000,0.0000,0.0000


## 7. Seller state analysis
This section summarizes seller count, orders, items, and revenue by seller state.

In [39]:
seller_state_query = """
SELECT
    seller_state,
    COUNT(DISTINCT seller_id) AS total_sellers,
    SUM(total_orders) AS total_orders,
    SUM(total_items_sold) AS total_items_sold,
    ROUND(SUM(seller_revenue)::numeric, 2) AS total_revenue,
    ROUND(AVG(seller_revenue)::numeric, 2) AS avg_revenue_per_seller
FROM vw_seller_summary
GROUP BY seller_state
ORDER BY total_revenue DESC;
"""

df_seller_state = pd.read_sql(seller_state_query, engine)
df_seller_state

,seller_state,total_sellers,total_orders,total_items_sold,total_revenue,avg_revenue_per_seller
0,SP,1849,70950.0,80342.0,8753396.21,4734.12
1,PR,349,7717.0,8671.0,1261887.21,3615.72
2,MG,244,7942.0,8827.0,1011564.74,4145.76
3,RJ,171,4355.0,4818.0,843984.22,4935.58
4,SC,190,3672.0,4075.0,632426.07,3328.56
5,RS,129,1991.0,2199.0,378559.54,2934.57
6,BA,19,569.0,643.0,285561.56,15029.56
7,DF,30,824.0,899.0,97749.48,3258.32
8,PE,9,406.0,448.0,91493.85,10165.98
9,GO,40,463.0,520.0,66399.21,1659.98


## 8. Top seller cities
This section identifies the seller cities contributing the most revenue.

In [40]:
seller_city_query = """
SELECT
    seller_city,
    seller_state,
    COUNT(DISTINCT seller_id) AS total_sellers,
    SUM(total_orders) AS total_orders,
    ROUND(SUM(seller_revenue)::numeric, 2) AS total_revenue
FROM vw_seller_summary
GROUP BY seller_city, seller_state
ORDER BY total_revenue DESC
LIMIT 25;
"""

df_seller_city = pd.read_sql(seller_city_query, engine)
df_seller_city

,seller_city,seller_state,total_sellers,total_orders,total_revenue
0,sao paulo,SP,694,24666.0,2702878.14
1,ibitinga,SP,49,6716.0,624592.94
2,curitiba,PR,124,2657.0,467821.52
3,rio de janeiro,RJ,93,2189.0,358126.92
4,guarulhos,SP,50,2073.0,329494.38
5,ribeirao preto,SP,52,2019.0,275976.44
6,itaquaquecetuba,SP,9,1243.0,230568.12
7,guariba,SP,1,1132.0,229472.63
8,santo andre,SP,45,2711.0,228561.60
9,lauro de freitas,BA,2,359.0,225525.05


## 9. Seller review performance
This section measures average review score by seller using order-item to order-review mapping.

In [41]:
seller_review_query = """
SELECT
    seller_id,
    orders AS total_orders,
    ROUND(avg_review_score::numeric, 4) AS avg_review_score
FROM vw_seller_performance_summary
ORDER BY avg_review_score DESC NULLS LAST;
"""

df_seller_review = pd.read_sql(seller_review_query, engine)
df_seller_review

,seller_id,total_orders,avg_review_score
0,3d68c634a99a1ba46dbca3967a69c623,2,5.0
1,2bd05d410a8fd26dc4184a15f4f2f588,6,5.0
2,2bdb95a56a36ebbc6640337ac5eac174,1,5.0
3,43753b27d77860f1654aa72e251a7878,1,5.0
4,2c00c85d30361cd2ced2969cffbbffa3,1,5.0
...,...,...,...
2965,20f0aeea30bc3b8c4420be8ced4226c0,1,NaN
2966,4e2627090e6e5b9fabba883a37897683,1,NaN
2967,80ceebb4ee9b31afb6c6a916a574a1e2,1,NaN
2968,bcd2d7510d58e293f20fad6438c1b314,1,NaN


## 10. Seller delivery performance
This section evaluates how often sellers are associated with late-delivered orders.

In [42]:
seller_delivery_query = """
SELECT
    oi.seller_id,
    s.seller_city,
    s.seller_state,
    COUNT(DISTINCT oi.order_id) AS total_orders,
    COUNT(
        DISTINCT CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN oi.order_id
        END
    ) AS late_orders,
    ROUND(
        100.0 * COUNT(
            DISTINCT CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN oi.order_id
            END
        ) / NULLIF(COUNT(DISTINCT oi.order_id), 0),
        2
    ) AS late_order_rate_pct
FROM olist_order_items oi
LEFT JOIN olist_sellers s
    ON oi.seller_id = s.seller_id
LEFT JOIN olist_orders o
    ON oi.order_id = o.order_id
GROUP BY
    oi.seller_id,
    s.seller_city,
    s.seller_state
HAVING COUNT(DISTINCT oi.order_id) >= 10
ORDER BY late_order_rate_pct DESC, total_orders DESC
LIMIT 50;
"""

df_seller_delivery = pd.read_sql(seller_delivery_query, engine)
df_seller_delivery

,seller_id,seller_city,seller_state,total_orders,late_orders,late_order_rate_pct
0,b1b3948701c5c72445495bd161b83a4c,sao paulo,SP,18,9,50.00
1,973f21788dfab357250f69a8dcb7ddee,claudio,MG,10,5,50.00
2,26e2c91ef821e1ff8985f408788fe35b,ibitinga,SP,12,5,41.67
3,f76a3b1349b6df1ee875d1f3fa4340f0,sao paulo,SP,24,9,37.50
4,d9e7e7778b32987280a6f2cb9a39c57d,sao paulo,SP,11,4,36.36
5,b0b346d3a89f5eb4c2968af3f083cd43,campinas,SP,11,4,36.36
6,02d35243ea2e497335cd0f076b45675d,natal,RN,14,5,35.71
7,0725b8c0f3f906e58f70cbe76b7c748c,guarulhos,SP,17,6,35.29
8,cb41bfbcbda0aea354a834ab222f9a59,sao paulo,SP,12,4,33.33
9,e64d65bc8dbec2accda90c58de5d1246,rio claro,SP,12,4,33.33


## 11. Seller segmentation
This section groups sellers into simple revenue-based performance segments.

In [43]:
seller_segment_query = """
SELECT
    CASE
        WHEN seller_revenue < 1000 THEN 'Small'
        WHEN seller_revenue < 10000 THEN 'Mid'
        WHEN seller_revenue < 50000 THEN 'Large'
        ELSE 'Top'
    END AS seller_segment,
    COUNT(*) AS seller_count,
    ROUND(AVG(seller_revenue)::numeric, 2) AS avg_revenue,
    ROUND(AVG(total_orders)::numeric, 2) AS avg_orders
FROM vw_seller_summary
GROUP BY
    CASE
        WHEN seller_revenue < 1000 THEN 'Small'
        WHEN seller_revenue < 10000 THEN 'Mid'
        WHEN seller_revenue < 50000 THEN 'Large'
        ELSE 'Top'
    END
ORDER BY seller_count DESC;
"""

df_seller_segments = pd.read_sql(seller_segment_query, engine)
df_seller_segments

,seller_segment,seller_count,avg_revenue,avg_orders
0,Small,1667,327.10,4.03
1,Mid,1136,3555.09,29.95
2,Large,252,19812.97,138.98
3,Top,40,100373.07,606.33


## 12. Export seller outputs
Save all seller analysis outputs into the project `sql/outputs/` folder.

In [44]:
OUTPUT_DIR = Path(r"C:\Users\divya\Retail-Intelligence-Platform\sql\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_seller_kpis.to_csv(OUTPUT_DIR / "seller_kpi_summary_sql.csv", index=False)
df_seller_performance.to_csv(OUTPUT_DIR / "seller_performance_summary_sql.csv", index=False)
df_seller_review.to_csv(OUTPUT_DIR / "seller_review_summary_sql.csv", index=False)

# keep your existing seller geography / segmentation exports too if those cells already exist
try:
    df_seller_state.to_csv(OUTPUT_DIR / "seller_state_summary_sql.csv", index=False)
except NameError:
    pass

try:
    df_seller_city.to_csv(OUTPUT_DIR / "seller_city_summary_sql.csv", index=False)
except NameError:
    pass

try:
    df_seller_segments.to_csv(OUTPUT_DIR / "seller_segment_summary_sql.csv", index=False)
except NameError:
    pass

print("Seller SQL outputs exported successfully.")
print(f"Saved to: {OUTPUT_DIR}")

Seller SQL outputs exported successfully.
Saved to: C:\Users\divya\Retail-Intelligence-Platform\sql\outputs


## 13. Summary

This notebook created the SQL seller analytics layer for the Retail Intelligence Platform.

### Outputs generated
- `seller_summary.csv`
- `top_sellers_revenue.csv`
- `top_sellers_orders.csv`
- `seller_state_summary.csv`
- `seller_city_summary.csv`
- `seller_review_summary.csv`
- `seller_delivery_summary.csv`
- `seller_segment_summary.csv`

### Next notebook
`04_sql_product_analysis.ipynb`